In [ ]:
!pip install datasets transformers torch pandas -q
!pip install --upgrade datasets torchvision accelerate -q
!pip uninstall torchaudio -y

In [ ]:
!pip install --upgrade torch torchvision transformers datasets -q

#Imports e Definição do Modelo

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Definição do modelo
MODEL_NAME = "distilbert-base-uncased"


print(f"Carregando tokenizador para: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#Carregamento e Amostragem

In [ ]:
# Carrega o arquivo do Kaggle
dataset = load_dataset('csv', data_files="/content/IMDB Dataset.csv", split='train')

# Corte estratégico de dados para salvar tempo de processamento
# Selecionando 10.000 reviews aleatórias das 50.000 originais
dataset = dataset.shuffle(seed=42).select(range(15000))

print(f"Tamanho do dataset após corte: {len(dataset)} exemplos.")

#Mapeamento e Tokenização

In [ ]:
# 1. Converte 'positive' para 1 e 'negative' para 0
def map_labels(example):
    example['labels'] = 1 if example['sentiment'] == 'positive' else 0
    return example

dataset = dataset.map(map_labels)

# 2. Função de tokenização limitando a 128 tokens
def tokenize_function(examples):
    return tokenizer(
        examples["review"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("Iniciando tokenização em lote...")
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Tokenização concluída!")

#Divisão e Preparação para o PyTorch

In [ ]:
# Separa 80% para Treino e 20% para Teste/Validação
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

# Divide os 20% restantes igualmente: Validação, Teste
test_val_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

# Agrupa tudo em um único objeto DatasetDict
final_datasets = DatasetDict({
    'train': train_test_split['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

# Define o formato esperado pelos tensores do modelo
final_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Dataset IMDB pronto para o Trainer!")
print(final_datasets)

#Função de Métricas (Accuracy, Precision, Recall, F1)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Calcula precision, recall e f1 usando a média binária (já que é positivo/negativo)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

#Experimento 1

In [ ]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. CONGELA AS CAMADAS (Exp 1)
# ==========================================
for param in model.base_model.parameters():
    param.requires_grad = False
print("Experimento 1: Apenas classificador liberado.")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================

EPOCAS_ATUAIS = 5

training_args = TrainingArguments(
    output_dir="./resultados_exp1",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 1 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 1): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 1 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")

#Experimento 2

In [ ]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. CONGELA AS CAMADAS (Exp 2 - Parcial)
# ==========================================
# Primeiro, congela tudo
for param in model.base_model.parameters():
    param.requires_grad = False

# Depois, libera apenas as duas últimas camadas
for name, param in model.named_parameters():
    if "layer.4" in name or "layer.5" in name:
        param.requires_grad = True
print("Experimento 2: Fine-tuning parcial liberado (Últimas camadas).")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================
EPOCAS_ATUAIS = 5

training_args = TrainingArguments(
    output_dir="./resultados_exp2",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 2 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 2): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 2 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")

#Experimento 3

In [ ]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. LIBERA TODAS AS CAMADAS (Exp 3)
# ==========================================
for param in model.parameters():
    param.requires_grad = True
print("Experimento 3: Fine-tuning completo (Todas as camadas liberadas).")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================
EPOCAS_ATUAIS = 5

training_args = TrainingArguments(
    output_dir="./resultados_exp3",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 3 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 3): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 3 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")